# Launching the tostadas pipeline (measles VADR)

The big one for submissions. **tostadas** is CDC's Nextflow pipeline for preparing pathogen genome submissions to NCBI databases (GenBank + SRA). After you've assembled a measles consensus genome from viralrecon (notebook 03), this notebook runs tostadas to validate it, annotate it via VADR, and produce a submission-ready package.

We're on the `feature/measles-vadr` branch of `CDCgov/tostadas`. That branch targets measles specifically (loads measles VADR models from `vadr-models-mev`) and has the fixes needed to run on GCP Batch.

**Time estimates:**

- **Dry-run on the demo data**: 15-30 minutes cold. The test profile sets `dry_run = true`, so no submission goes to NCBI — just validation + annotation. Most of the wall clock is GCP Batch allocating Spot VMs and pulling the VADR container (large, ~1 GB).
- **Subsequent runs (cached)**: 5-10 minutes.

**Cost**: cents. Single-sample test data on Spot VMs.

**Real submission to NCBI is commented out** at the bottom. Real submission requires a configured BioProject and BioSample on NCBI's portal (one-time setup outside this notebook).

**Known upstream issue (2026-07-02):** `PREP_SUBMISSION` and `UPDATE_SUBMISSION` in the `feature/measles-vadr` branch pass file paths as `val` inputs instead of `path` inputs, so Nextflow doesn't stage the files into the container. On a `gs://` workDir the paths don't resolve and both steps fail with `FileNotFoundError`. The rest of the pipeline (metadata validation, VADR annotation, GenBank cleanup) runs cleanly on GCP Batch and infrastructure is fully validated end-to-end through those steps. Tracked with Glen for an upstream fix in `modules/local/prep_submission/main.nf` (lines 47-53) and `modules/local/update_submission/main.nf` (lines 36-43). Once patched, `-resume` continues from where it stopped without re-running upstream tasks.


> **Kernel:** When JupyterLab prompts "Select Kernel," pick **`Python 3 (Local)`** (under "Start python Kernel"). Don't pick `Python 3 (ipykernel) (Local)`, PyTorch, or TensorFlow — those are missing libraries this notebook needs and will fail at the first GCS read with `ModuleNotFoundError`. If you already see `Python 3 (Local)` in the top-right of this tab, you're good to go.


## Tools you'll use

This notebook combines Nextflow (workflow engine), CDC's tostadas pipeline (NCBI submission packaging), VADR (NCBI's viral annotation tool), and GCP Batch (compute backend). If any of those names are new, [05-reference.ipynb](05-reference.ipynb) has paragraphs on Nextflow, nf-core, and GCP Batch; the tostadas + VADR specifics are explained inline below.


In [ ]:
import subprocess

DETECTED_PROJECT = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True
).strip()

# Most values auto-detected from the Workbench environment.
PROJECT_ID    = DETECTED_PROJECT                              # @param {type:"string"}
REGION        = "us-central1"                                 # @param {type:"string"}

# Buckets — auto-detected after we list visible buckets below.
RESULTS_BUCKET = ""                                           # @param {type:"string"}
WORK_BUCKET    = ""                                           # @param {type:"string"}

# Custom VPC network + subnet. APGAP projects don't have a default VPC,
# so Batch needs to know which network to put VMs on. Left blank so
# auto-detect fires against the current project's VPC. If auto-detect
# can't find one (SA lacks `compute.subnetworks.list`, unusual VPC
# setup, etc.), set them manually here.
NETWORK        = ""                                           # @param {type:"string"}
SUBNETWORK     = ""                                           # @param {type:"string"}

# NCBI credentials. Required ONLY for real submission. Leave blank for
# the dry-run path this notebook demonstrates — the `test` profile sets
# `dry_run = true` which skips the actual push to NCBI.
NCBI_USERNAME  = ""                                           # @param {type:"string"}
NCBI_PASSWORD  = ""                                           # @param {type:"string"}

# Glen's pre-loaded measles test data. The bucket has:
#   - AF266288.fasta (a public measles genome)
#   - measles_demo_metadata.xlsx (with fasta_path set to a gs:// URI,
#     and ncbi-spuid / ncbi-bioproject pre-filled with demo values so
#     the genbank dry-run "just works")
TEST_BUCKET     = "gs://measles-analytical-dataset-33-1782282025-5e70a8"  # @param {type:"string"}
METADATA_FILE   = "measles_demo_metadata.xlsx"                            # @param {type:"string"}

# Tostadas pipeline repo + branch. The measles-vadr branch contains the
# GCP Batch compatibility fix needed to run under cloud executors.
TOSTADAS_REPO   = "https://github.com/CDCgov/tostadas.git"    # @param {type:"string"}
TOSTADAS_BRANCH = "feature/measles-vadr"                      # @param {type:"string"}

In [ ]:
import gcsfs
import subprocess

active_account = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()
print(f"GCP project: {PROJECT_ID}")
print(f"Region:      {REGION}")
print(f"Identity:    {active_account}\n")

fs = gcsfs.GCSFileSystem()
all_buckets = fs.ls("")

if not RESULTS_BUCKET:
    out = [b.rstrip("/") for b in all_buckets if "seqera-output" in b]
    if len(out) > 1:
        print(f"⚠ Multiple seqera-output buckets visible ({len(out)}): {out}")
        print(f"  Auto-selecting the first one. Override RESULTS_BUCKET above if that's wrong.")
    if out:
        RESULTS_BUCKET = f"gs://{out[0]}/"
        print(f"Auto-selected RESULTS_BUCKET: {RESULTS_BUCKET}")
if not WORK_BUCKET:
    WORK_BUCKET = RESULTS_BUCKET.rstrip("/") + "/tostadas-work/"
    print(f"Auto-selected WORK_BUCKET:    {WORK_BUCKET}")

# Auto-detect the project's custom VPC. Same subnet-first pattern as
# nb 03 (the SA can list subnets but not networks, so we query subnets
# and extract the parent network from each row's network field).
# Collect stderr from each empty-result query so the RuntimeError below
# can surface *why* auto-detect came back empty (permission denied vs.
# no VPC in the project vs. wrong region), instead of a silent (none).
def _detect_network_subnet(network, subnet):
    diagnostics = []
    if not subnet:
        r = subprocess.run(
            ["gcloud", "compute", "networks", "subnets", "list",
             f"--project={PROJECT_ID}",
             f"--regions={REGION}",
             "--format=value(name,network)",
             "--limit=1"],
            capture_output=True, text=True,
        )
        rows = r.stdout.strip().splitlines()
        if rows:
            parts = rows[0].split()
            subnet = parts[0]
            if not network and len(parts) > 1:
                network = parts[1].rsplit("/", 1)[-1]
        elif r.stderr.strip():
            diagnostics.append(f"[subnets list stderr]\n{r.stderr.strip()}")
    if not network:
        r = subprocess.run(
            ["gcloud", "compute", "networks", "list",
             f"--project={PROJECT_ID}",
             "--format=value(name)",
             "--limit=1"],
            capture_output=True, text=True,
        )
        if r.stdout.strip():
            network = r.stdout.strip().splitlines()[0]
        elif r.stderr.strip():
            diagnostics.append(f"[networks list stderr]\n{r.stderr.strip()}")
    return network, subnet, diagnostics

NETWORK, SUBNETWORK, _detect_diagnostics = _detect_network_subnet(NETWORK, SUBNETWORK)
print(f"Network:      {NETWORK or '(none)'}")
print(f"Subnetwork:   {SUBNETWORK or '(none)'}")

if not NETWORK or not SUBNETWORK:
    diag_block = ("\n\n" + "\n\n".join(_detect_diagnostics)) if _detect_diagnostics else ""
    raise RuntimeError(
        "Couldn't auto-detect a VPC + subnet, and NETWORK or SUBNETWORK is "
        "empty in the parameter cell above. Ask your project lead for the "
        "network and subnet names for this project, then paste them above "
        "and re-run." + diag_block
    )


## Installing Nextflow and Java

Nextflow runs on the JVM. We install both via whichever conda-compatible package manager the Workbench has on PATH (`micromamba`, `conda`, or `mamba` — the image varies). See [05-reference.ipynb](05-reference.ipynb) for the version-pinning rationale.

Where the install lands depends on the package manager:

- **`micromamba`**: installs into a per-user env at `~/nf-env`, with its own root prefix at `~/.micromamba`. Some Workbench images (seen on `measles-apgap-project-0f70` and similar) make the shared `/opt/micromamba/pkgs/cache` read-only for the `jupyter` user, which breaks a system-wide `-n base` install. A per-user env sidesteps that entirely — everything writes under `$HOME` where you own the directory.
- **`conda` / `mamba`**: installs into the active env (typically `base`), which has been writable on every image tested.

Either way, the cell is a no-op if the pinned versions are already there. If the install fails (network glitch, bioconda channel hiccup), just re-run the cell.

In [ ]:
import os
import shutil
import subprocess

# Workbench images vary: newer ones ship `micromamba`, older ones `conda`.
# `mamba` is sometimes a thin wrapper that can behave unpredictably with
# conda-style flags, so it sits last in the preference order.
for _mgr in ("micromamba", "conda", "mamba"):
    if shutil.which(_mgr):
        PKG_MANAGER = _mgr
        break
else:
    raise RuntimeError(
        "No conda-compatible package manager (micromamba, conda, mamba) "
        "found in PATH. This notebook needs one to install Nextflow + JDK "
        "from the bioconda / conda-forge channels."
    )

# Install path differs between package managers:
#
# - `micromamba`: install into a per-user env at ~/nf-env with its own
#   root-prefix at ~/.micromamba. Some Workbench images lock down
#   /opt/micromamba/pkgs/cache for the jupyter user, which breaks
#   `micromamba install -n base` (needs to write package tarballs to the
#   shared cache). A per-user env writes everything under $HOME.
# - `conda` / `mamba`: install into the active env (typically base).
if PKG_MANAGER == "micromamba":
    NF_ENV = os.path.expanduser("~/nf-env")
    MAMBA_ROOT = os.path.expanduser("~/.micromamba")
    verb = "install" if os.path.exists(os.path.join(NF_ENV, "conda-meta")) else "create"
    _install_cmd = [
        "micromamba", verb, "-y",
        "--prefix", NF_ENV,
        "--root-prefix", MAMBA_ROOT,
    ]
else:
    _install_cmd = [PKG_MANAGER, "install", "-y"]

_install_cmd += ["-c", "bioconda", "-c", "conda-forge",
                 "openjdk=17", "nextflow=23.10"]

print(f"Pinning Nextflow 23.10 + JDK 17 via {PKG_MANAGER}...")
subprocess.run(_install_cmd, check=True)

# For micromamba's per-user env, add the env's bin dir to PATH so the
# next cell (and everything after) finds the nextflow + java binaries.
# conda/mamba install into the active env which is already on PATH.
if PKG_MANAGER == "micromamba":
    env_bin = os.path.join(NF_ENV, "bin")
    if env_bin not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] = env_bin + os.pathsep + os.environ["PATH"]
    print(f"Added {env_bin} to PATH")

print("Install complete.")

In [ ]:
import subprocess

print("=== java -version ===")
subprocess.run(["java", "-version"], check=True)
print("\n=== nextflow -version ===")
subprocess.run(["nextflow", "-version"], check=True)


## Get the tostadas pipeline

Clone the measles-vadr branch into the notebook directory. The cell is idempotent: if `tostadas/` already exists it skips the clone and just confirms the branch.


In [ ]:
import os
import subprocess

TOSTADAS_DIR = "tostadas"

if os.path.exists(TOSTADAS_DIR):
    print(f"tostadas/ already exists, skipping clone.")
    branch = subprocess.run(
        ["git", "-C", TOSTADAS_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f"Current branch: {branch}")
else:
    print(f"Cloning {TOSTADAS_REPO} (branch {TOSTADAS_BRANCH})...")
    subprocess.run(
        ["git", "clone", "-b", TOSTADAS_BRANCH, "--depth", "1",
         TOSTADAS_REPO, TOSTADAS_DIR],
        check=True,
    )
    print("Clone complete.")


## Configure NCBI submission credentials

Tostadas reads NCBI credentials from `tostadas/conf/submission_config.yaml`. The next cell substitutes your `NCBI_USERNAME` and `NCBI_PASSWORD` from the parameter cell into the right fields of that YAML.

**For the dry-run path this notebook demonstrates, the credentials can be blank.** The `test` profile sets `dry_run = true`, which means tostadas validates and annotates the genome but never pushes anything to NCBI's submission endpoints.

**For a real submission** you need:

1. An NCBI account at https://account.ncbi.nlm.nih.gov
2. A BioProject + BioSample set up at https://submit.ncbi.nlm.nih.gov, with the BioProject accession written into the metadata spreadsheet's `ncbi-bioproject` column
3. Fill in `NCBI_USERNAME` and `NCBI_PASSWORD` in the parameter cell above before running
4. Update the other fields in `tostadas/conf/submission_config.yaml` (Submitting_Org, contact email, BioSample_package, etc.) to match your organization

**Important**: this writes credentials in plain text to a file on the Workbench VM (`tostadas/conf/submission_config.yaml`). The VM is scoped to this APGAP project's users, but treat the file accordingly — don't commit it to git or share screenshots of it.


In [ ]:
import os

CONFIG_PATH = os.path.join(TOSTADAS_DIR, "conf", "submission_config.yaml")

with open(CONFIG_PATH) as f:
    config = f.read()

# YAML keys are at top of the file: `NCBI_username:` / `NCBI_password:`
# (both blank by default in the upstream template).
if NCBI_USERNAME:
    config = config.replace("NCBI_username:\n", f"NCBI_username: {NCBI_USERNAME}\n")
if NCBI_PASSWORD:
    config = config.replace("NCBI_password:\n", f"NCBI_password: {NCBI_PASSWORD}\n")

with open(CONFIG_PATH, "w") as f:
    f.write(config)

if NCBI_USERNAME and NCBI_PASSWORD:
    print(f"Wrote NCBI credentials to {CONFIG_PATH}")
else:
    print(f"Left credentials blank in {CONFIG_PATH} (dry-run mode).")

print(
    "Other config fields (Submitting_Org, contact, BioSample_package, etc.) "
    "remain at upstream defaults. Update them in the YAML before a real "
    "submission."
)


## Pointing Nextflow at GCP Batch

Same approach as notebook 03. We write our own `nextflow.config` that adds a `gcb` profile telling Nextflow to use the `google-batch` executor, with the project's custom VPC + service account. Tostadas itself doesn't ship a Google Batch profile, so we layer our config on top via `-c nextflow.config` at launch.


In [ ]:
config_content = f"""
profiles {{
    gcb {{
        // Merge ALL process-scope settings into ONE block. Same parser
        // gotcha as nb 03: mixing dot-syntax (process.executor = ...) with
        // a block-syntax `process {{ ... }}` in the same profile causes
        // Nextflow's legacy parser to silently drop the dot-syntax.
        process {{
            executor      = 'google-batch'
            cpus          = 2
            memory        = '4 GB'
            errorStrategy = {{ task.attempt <= 2 ? 'retry' : 'finish' }}
            maxRetries    = 2

            // VADR annotation is the memory-hungry step (cmsearch +
            // cmalign against the measles covariance model). Tostadas
            // doesn't set explicit resources for it (all withLabel:
            // process_* blocks are commented in tostadas/conf/modules.
            // config), so without an override it inherits Nextflow's
            // default 6 GB and gets SIGKILL'd by the container cgroup
            // limit. 16 and 32 GB both OOMed on measles test data; 64 GB
            // was the first level that let cmalign complete. VADR's own
            // error message flags this: "may have run out of available
            // memory, especially if you see a 'Killed' message."
            //
            // The afterScript is a diagnostic safety net. VADR runs its
            // inner annotation script with `2> AF266288_mev/*.sh.err`,
            // writing to the container's local FS. That file gets
            // destroyed with the container and never staged out, so
            // failures produce no useful stderr in .command.err. This
            // find dumps any inner .sh.err files to stdout before exit;
            // Nextflow captures it into .command.out which IS staged.
            withName: 'VADR_ANNOTATION' {{
                cpus        = 4
                memory      = '64 GB'
                afterScript = 'find . -name "*.sh.err" -print -exec cat {{}} \\\\; 2>&1 || true'
            }}
        }}

        google {{
            project  = '{PROJECT_ID}'
            location = '{REGION}'
            batch {{
                serviceAccountEmail = '{active_account}'
                network    = 'projects/{PROJECT_ID}/global/networks/{NETWORK}'
                subnetwork = 'projects/{PROJECT_ID}/regions/{REGION}/subnetworks/{SUBNETWORK}'
                spot = true
            }}
        }}

        workDir = '{WORK_BUCKET}'
    }}
}}
"""

CONFIG_LOCAL = "nextflow.config"
with open(CONFIG_LOCAL, "w") as f:
    f.write(config_content)

print(f"Wrote {CONFIG_LOCAL}.\n")
print(open(CONFIG_LOCAL).read())


## Step 1: dry run with `-preview`

Like nb 03, we do a quick sanity check first. `-preview` parses the pipeline + config + profile combination without launching any GCP Batch jobs. Takes ~10 seconds. If profile stacking or config syntax is wrong, we find out cheaply.


In [ ]:
import subprocess

result = subprocess.run(
    ["nextflow", "run", os.path.join(TOSTADAS_DIR, "main.nf"),
     "-profile", "test,docker,gcb",
     "--species", "virus",
     "-preview",
     "-c", "nextflow.config"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("DRY RUN FAILED — fix config before continuing:")
    print(result.stderr)


## Step 2: dry-run on the measles demo data

This launches the measles workflow against Glen's pre-loaded test data. Each pipeline step becomes a GCP Batch job. Nothing gets pushed to NCBI: `--dry_run true` keeps the submission step in validate-only mode.

We pass the metadata file's GCS URI directly to `--updated_meta_path`. Nextflow's `path` input handler downloads the file into each task's work directory automatically, so we don't need a local copy. Same staging behavior applies to the FASTA path stored inside the metadata spreadsheet — Glen pre-set `fasta_path` to a `gs://` URI.

Expected duration: 15-30 minutes cold (most of the wall clock is the VADR container pull, which is large). 5-10 minutes when re-running with `-resume` against a warm cache.

Expected cost: a few cents.

Watch on GCP: https://console.cloud.google.com/batch/jobs


In [ ]:
import select
import subprocess

results_outdir = RESULTS_BUCKET.rstrip("/") + "/tostadas-test-results"
metadata_uri = f"{TEST_BUCKET.rstrip('/')}/{METADATA_FILE}"

print(f"Launching tostadas dry-run...")
print(f"  metadata: {metadata_uri}")
print(f"  outdir:   {results_outdir}\n")

process = subprocess.Popen(
    ["nextflow", "run", os.path.join(TOSTADAS_DIR, "main.nf"),
     "-profile", "measles,docker,gcb",
     "-c", "nextflow.config",
     "-resume",
     "--workflow", "genbank",
     "--updated_meta_path", metadata_uri,
     "--submission_config", os.path.join(TOSTADAS_DIR, "conf", "submission_config.yaml"),
     "--outdir", results_outdir,
     "--dry_run", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

# Same select-based stream loop as nb 03 (avoids the Nextflow JVM
# zombie-threads hang where the stdout pipe never sees EOF).
try:
    while True:
        rlist, _, _ = select.select([process.stdout], [], [], 1.0)
        if rlist:
            line = process.stdout.readline()
            if line:
                print(line, end="", flush=True)
        elif process.poll() is not None:
            break
finally:
    process.stdout.close()

print(f"\nNextflow exited with code {process.returncode}")


## What tostadas produced

The pipeline writes outputs to the GCS path passed as `--outdir`. Notable artifacts:

- **`*.sqn` files** — the validated submission packages, one per sample. Even in dry-run mode these get generated; they're what would have been pushed to GenBank in a real submission.
- **VADR reports** (`*.vadr.*`) — annotation results showing which gene features VADR detected on the consensus.
- **Metadata reports + validation summaries** — useful when troubleshooting a failed run.

The cell below uses `gcsfs.find()` to locate the most interesting outputs. If nothing shows up, check the Nextflow log above for the actual failure.


In [ ]:
import gcsfs

fs = gcsfs.GCSFileSystem()
results_prefix = results_outdir.replace("gs://", "").rstrip("/")

try:
    all_files = list(fs.find(results_prefix))
    sqn_files = [p for p in all_files if p.endswith(".sqn")]
    vadr_files = [p for p in all_files if "vadr" in p.rsplit("/", 1)[-1].lower()]

    print(f"Total files in gs://{results_prefix}: {len(all_files)}\n")

    def _show(label, paths, limit=10):
        print(f"{label} ({len(paths)}):")
        for p in paths[:limit]:
            size_kb = fs.info(p).get("size", 0) / 1024
            print(f"  {size_kb:>7.1f} KB  gs://{p}")
        if len(paths) > limit:
            print(f"  ... and {len(paths) - limit} more")
        print()

    _show("Submission packages (.sqn)", sqn_files)
    _show("VADR annotation outputs", vadr_files)

    if not sqn_files and not vadr_files:
        print("No .sqn or VADR outputs found. Either the pipeline failed "
              "early or the outdir is in a different location — check "
              "the Nextflow log above.")
except FileNotFoundError:
    print(f"Results prefix not found at gs://{results_prefix}.")
    print("Pipeline may have failed before writing outputs — check the "
          "Nextflow log above.")


## Real submission (commented out)

The next cell would launch tostadas with real submission to NCBI's production endpoints. It's commented out on purpose. **Uncomment only after** all of these are true:

1. `NCBI_USERNAME` and `NCBI_PASSWORD` in the parameter cell are real credentials for an account authorized to submit
2. You have a **BioProject set up** at https://submit.ncbi.nlm.nih.gov and its accession is in the metadata spreadsheet's `ncbi-bioproject` column
3. You've reviewed and updated the other fields in `tostadas/conf/submission_config.yaml` (Submitting_Org, contact email, BioSample_package, etc.)
4. The dry-run above completed successfully on the same data

A real submission costs cents on GCP Batch but submits a **permanent record to NCBI's public databases** — review the metadata + the generated `.sqn` carefully before running.


In [ ]:
# import select
# import subprocess
#
# real_outdir = RESULTS_BUCKET.rstrip("/") + "/tostadas-real-submission"
#
# process = subprocess.Popen(
#     ["nextflow", "run", os.path.join(TOSTADAS_DIR, "main.nf"),
#      "-profile", "measles,docker,gcb",
#      "-c", "nextflow.config",
#      "-resume",
#      "--workflow", "genbank",
#      "--updated_meta_path", f"{TEST_BUCKET.rstrip('/')}/{METADATA_FILE}",
#      "--submission_config", os.path.join(TOSTADAS_DIR, "conf", "submission_config.yaml"),
#      "--outdir", real_outdir,
#      "--prod_submission", "true",   # switch to NCBI's production endpoint
#      "--submission", "true",         # actually push
#      "--dry_run", "false"],
#     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
# )
#
# # Same select-based stream loop as the dry-run cell above.
# import select
# try:
#     while True:
#         rlist, _, _ = select.select([process.stdout], [], [], 1.0)
#         if rlist:
#             line = process.stdout.readline()
#             if line:
#                 print(line, end="", flush=True)
#         elif process.poll() is not None:
#             break
# finally:
#     process.stdout.close()
#
# print(f"\nNextflow exited with code {process.returncode}")


## Where to go next

- **Upstream consensus generation** — [03-launch-a-pipeline.ipynb](03-launch-a-pipeline.ipynb) (viralrecon produces the consensus FASTAs you'd feed into this notebook)
- **File format reference** — [05-reference.ipynb](05-reference.ipynb) (FASTA, VCF, GFF, etc.)
- **Back to the** [getting-started overview](01-getting-started.ipynb)

External docs:

- [tostadas repo](https://github.com/CDCgov/tostadas)
- [measles-vadr branch](https://github.com/CDCgov/tostadas/tree/feature/measles-vadr)
- [VADR (NCBI's viral annotator)](https://github.com/ncbi/vadr)
- [NCBI submission portal](https://submit.ncbi.nlm.nih.gov)
